# Common Mind Survey - Exploratory Data Analysis

This notebook performs initial EDA on the Typeform survey export data.

**Setup:** Run the cells below to install packages and connect to your data.

In [ ]:
# Install required packages (run once)
!pip install -q pandas numpy matplotlib seaborn plotly openpyxl openai

In [ ]:
# Mount Google Drive to access your data files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Plot settings
sns.set_style('whitegrid')
sns.set_palette('husl')

print('Libraries loaded successfully!')

## 1. Load Data

**Option A:** Load from Google Drive (recommended)  
**Option B:** Upload file directly to Colab

In [ ]:
# OPTION A: Load from Google Drive
# Update this path to match where you saved your Typeform export in Drive
DATA_PATH = '/content/drive/MyDrive/CommonMind/'

# List files in the folder
import os
if os.path.exists(DATA_PATH):
    print('Files in folder:')
    for f in os.listdir(DATA_PATH):
        print(f'  - {f}')
else:
    print(f'Folder not found: {DATA_PATH}')
    print('Create this folder in Drive and upload your Typeform export, or update the path.')

In [ ]:
# OPTION B: Upload file directly (alternative to Drive)
# Uncomment to use:

# from google.colab import files
# uploaded = files.upload()  # This will prompt you to select a file
# filename = list(uploaded.keys())[0]
# df = pd.read_csv(filename)  # or pd.read_excel(filename)

In [ ]:
# Load the survey data from Drive
# Update the filename to match your Typeform export

FILENAME = 'your_typeform_export.csv'  # <-- UPDATE THIS

filepath = DATA_PATH + FILENAME
if FILENAME.endswith('.csv'):
    df = pd.read_csv(filepath)
else:
    df = pd.read_excel(filepath)

print(f'Loaded {len(df)} responses')
df.head()

## 2. Data Overview

In [ ]:
print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nColumn names:\n')
for i, col in enumerate(df.columns, 1):
    print(f'  {i}. {col}')

In [ ]:
# Data types and missing values
df.info()

In [ ]:
# Summary statistics
df.describe(include='all')

## 3. Missing Data Analysis

In [ ]:
# Missing values summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df[missing_df['Missing'] > 0].sort_values('Percent', ascending=False)

In [ ]:
# Visualize missing data
if missing_df['Missing'].sum() > 0:
    plt.figure(figsize=(12, 6))
    missing_data = missing_df[missing_df['Missing'] > 0].sort_values('Percent', ascending=True)
    plt.barh(range(len(missing_data)), missing_data['Percent'])
    plt.yticks(range(len(missing_data)), missing_data.index)
    plt.xlabel('% Missing')
    plt.title('Missing Data by Column')
    plt.tight_layout()
    plt.show()
else:
    print('No missing data!')

## 4. Response Distributions

In [ ]:
# Identify categorical columns (for visualization)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns ({len(categorical_cols)}):')
for col in categorical_cols:
    unique_count = df[col].nunique()
    print(f'  - {col}: {unique_count} unique values')

In [ ]:
# Visualize a categorical column
# Update COLUMN_NAME to explore different questions

COLUMN_NAME = categorical_cols[0] if categorical_cols else None  # <-- UPDATE THIS

if COLUMN_NAME:
    fig, ax = plt.subplots(figsize=(10, 6))
    df[COLUMN_NAME].value_counts().plot(kind='bar', ax=ax)
    plt.title(f'Response Distribution: {COLUMN_NAME}')
    plt.xlabel('')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 5. AI-Powered Analysis (Optional)

Use OpenAI to analyze open-ended text responses.

In [ ]:
# Set your OpenAI API key
import os
from getpass import getpass

# This will prompt you to enter your API key securely
os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')

In [ ]:
from openai import OpenAI

client = OpenAI()

def analyze_responses(responses, question=""):
    """Analyze open-ended survey responses using AI."""
    responses_text = "\n---\n".join(responses[:50])  # Limit to 50 for API costs
    
    prompt = f"""Analyze these survey responses{f' to the question: {question}' if question else ''}.

Provide:
1. Key themes and patterns
2. Common sentiments
3. Notable insights

Responses:
{responses_text}"""
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

In [ ]:
# Analyze a text column
# Update TEXT_COLUMN to the column containing open-ended responses

TEXT_COLUMN = None  # <-- UPDATE THIS (e.g., 'What feedback do you have?')

if TEXT_COLUMN and TEXT_COLUMN in df.columns:
    text_responses = df[TEXT_COLUMN].dropna().tolist()
    print(f'Analyzing {len(text_responses)} responses...\n')
    analysis = analyze_responses(text_responses, question=TEXT_COLUMN)
    print(analysis)
else:
    print('Set TEXT_COLUMN to a column name containing open-ended responses')

## 6. Next Steps

- [ ] Clean and preprocess the data
- [ ] Deep-dive into specific survey questions
- [ ] Cross-tabulate responses
- [ ] Generate insights report